# Website Scraping

This contains the code used the scrape the Common Data Set website\. The Terms & Conditions on the website allow for scraping as long as it is not for reupload\.

In [1]:
import json
import time
import pandas as pd
import requests

LIST_URL = "https://commondatasets.com/data/schools-2025-2026.json"
DETAIL_URL = ""
ID_FIELD = "id"          
DELAY = 2                

HEADERS = {"User-Agent": "Personal research script"}


def get_json(url):
    r = requests.get(url, headers=HEADERS, timeout=30)
    r.raise_for_status()
    return r.json()


def find_records(data):
    """The school list is sometimes wrapped, e.g. {"schools": [...]}. Unwrap it."""
    if isinstance(data, list):
        return data
    for value in data.values():
        if isinstance(value, list) and value and isinstance(value[0], dict):
            return value
    raise ValueError("No list of records found - inspect the JSON and adjust.")


def main():
    records = find_records(get_json(LIST_URL))
    print(f"Found {len(records)} schools in the list.")

    if DETAIL_URL:
        detailed = []
        for rec in records:
            school_id = rec.get(ID_FIELD)
            try:
                time.sleep(DELAY)
                detailed.append(get_json(DETAIL_URL.format(id=school_id)))
                print(f"ok    {school_id}")
            except Exception as e:
                print(f"error {school_id}: {e}")
                detailed.append(rec)  
        records = detailed

    with open("commondatasets_raw.json", "w") as f:   
        json.dump(records, f, indent=2)

    df = pd.json_normalize(records)                    
    df.to_csv("commondatasets_admissions.csv", index=False)
    print(f"Saved {len(df)} rows x {df.shape[1]} columns to commondatasets_admissions.csv")


if __name__ == "__main__":
    main()

Found 165 schools in the list.
Saved 165 rows x 158 columns to commondatasets_admissions.csv


<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=5504e1cb-e852-40a9-a959-1b5368c81868' target="_blank">

Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>